In [13]:
import numpy as np
import pandas as pd

In [16]:
def generate_ordinal_customer_data(n=500,noise=0.1):
    rng = np.random.default_rng(seed=42) # random number generator

    # Ordered categories
    education_levels = [
            'high_school',
            'bachelor',
            'masters',
            'phd'
    ]

    education = rng.choice(
        education_levels,
        size=n,
        p=[0.50, 0.30, 0.15, 0.05]
    )

    age = rng.integers(18, 60, size=n)

    income = rng.normal(
        loc=40000,
        scale=15000,
        size=n
    ).astype(int)

    income = np.clip(income, 8000, 150000)

    education_score = {
        'high_school': 0,
        'bachelor': 1,
        'masters': 2,
        'phd': 3
    }

    edu_numeric = np.array(
        [education_score[e] for e in education]
    )

    approval_probability = (
        0.2 # base approval probability(0.2)
        + 0.15 * edu_numeric # hugher education = higher approval chance
        + 0.00005 * income # higher income = higher approval chance
        - 0.02 * (age - 30).clip(min=0) # old age = after age = 30, reduces approval chance
    )

    approval_probability = np.clip(
        approval_probability, 0, 1
    )

    target = (
        rng.random(n) < approval_probability
    ).astype(int) # 1 = approved, 0 = reject

    noise_mask = rng.random(n) < noise
    target[noise_mask] == 1 - target[noise_mask]

    return pd.DataFrame(
        {
            'age': age,
            'income': income,
            'education_level': education,
            'approved': target
        }
    )

In [17]:
df = generate_ordinal_customer_data(1000)
df = df.to_csv('ordinal_customer_dataset.csv', index=None)